# KOT-V2-Mini: Russian TinyStories Weight Lab

Интерактивный эксперимент: маленький KOT-подобный Transformer с GQA, RoPE, RMSNorm, SwiGLU и MoE Top-2/4. После обучения ползунок меняет степень расхождения четырёх FFN-экспертов относительно их среднего веса. Каждое изменение пересчитывает validation loss и генерацию.

**Важно:** это учебный мини-эксперимент с символьным токенизатором, а не полноценная KOT-V2-2B. Для эксперимента нужен GPU runtime (T4 подходит). Датасет: `Dxniz/TinyStories-Multilingual`, только строки `language_code == 'ru'`.

In [ ]:
!pip -q install datasets ipywidgets matplotlib

import os, math, copy, random, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from datasets import load_dataset
from IPython.display import display, clear_output
import ipywidgets as widgets

assert torch.cuda.is_available(), 'Включи GPU: Runtime → Change runtime type → T4 GPU'
device = 'cuda'
print('GPU:', torch.cuda.get_device_name(0), '| PyTorch:', torch.__version__)
torch.manual_seed(42); np.random.seed(42); random.seed(42)


## 1. Загружаем русские TinyStories

В репозитории датасета есть поле `language_code`; выбираем только `ru`, а текст берём из `output`. Используем символьную токенизацию, чтобы эксперимент был лёгким и не требовал отдельного BPE-токенизатора.

In [ ]:
ds = load_dataset('Dxniz/TinyStories-Multilingual', split='train')
ru = ds.filter(lambda x: x['language_code'] == 'ru')
stories = [s.strip() for s in ru['output'] if isinstance(s, str) and len(s.strip()) >= 80]
print('Русских историй:', len(stories))
if len(stories) < 100:
    raise RuntimeError('Получено слишком мало русских историй. Проверь доступность датасета.')

random.Random(42).shuffle(stories)
split = max(1, int(len(stories) * 0.95))
train_stories, val_stories = stories[:split], stories[split:]
chars = sorted(set(''.join(train_stories + val_stories) + '\n'))
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for ch,i in stoi.items()}
vocab_size = len(chars)
train_text = '\n'.join(train_stories) + '\n'
val_text = '\n'.join(val_stories) + '\n'
train_ids = torch.tensor([stoi[c] for c in train_text], dtype=torch.long)
val_ids = torch.tensor([stoi[c] for c in val_text], dtype=torch.long)
print(f'Символов в словаре: {vocab_size} | train: {len(train_ids):,} | val: {len(val_ids):,}')
print('\nПример:', train_stories[0][:350])


## 2. Архитектура KOT-V2-Mini

4 слоя, hidden size 192, GQA (6 Q-heads / 2 KV-heads), RoPE, RMSNorm, SwiGLU и MoE из четырёх экспертов с Top-2 routing. Чтобы эксперты не были абсолютно одинаковыми, при старте их копии получают небольшой шум.

In [ ]:
# Быстрые параметры для T4. При желании можно менять до запуска обучения.
BLOCK_SIZE = 256
BATCH_SIZE = 32
MAX_STEPS = 1200
EVAL_EVERY = 100
EVAL_BATCHES = 12
LR = 3e-4
N_LAYER = 4
N_HEAD = 6
N_KV_HEAD = 2
N_EMBD = 192
N_EXPERTS = 4
TOP_K = 2
DROPOUT = 0.05

assert N_EMBD % N_HEAD == 0 and N_HEAD % N_KV_HEAD == 0
HEAD_DIM = N_EMBD // N_HEAD

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__(); self.weight = nn.Parameter(torch.ones(dim)); self.eps = eps
    def forward(self, x):
        return x * torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps).to(x.dtype) * self.weight

def rope(x):
    # x: [B, heads, T, head_dim]
    B,H,T,D = x.shape
    half = D // 2
    inv = 1.0 / (10000 ** (torch.arange(0, half, device=x.device, dtype=torch.float32) / half))
    pos = torch.arange(T, device=x.device, dtype=torch.float32)
    ang = torch.outer(pos, inv)
    cos, sin = ang.cos()[None,None,:,:], ang.sin()[None,None,:,:]
    a, b = x.float()[..., :half], x.float()[..., half:2*half]
    y = torch.cat((a*cos - b*sin, a*sin + b*cos), dim=-1)
    if D % 2: y = torch.cat((y, x.float()[..., -1:]), dim=-1)
    return y.to(x.dtype)

class GQAAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.q = nn.Linear(N_EMBD, N_HEAD*HEAD_DIM, bias=False)
        self.k = nn.Linear(N_EMBD, N_KV_HEAD*HEAD_DIM, bias=False)
        self.v = nn.Linear(N_EMBD, N_KV_HEAD*HEAD_DIM, bias=False)
        self.proj = nn.Linear(N_EMBD, N_EMBD, bias=False)
        self.drop = nn.Dropout(DROPOUT)
    def forward(self, x):
        B,T,C = x.shape
        q = self.q(x).view(B,T,N_HEAD,HEAD_DIM).transpose(1,2)
        k = self.k(x).view(B,T,N_KV_HEAD,HEAD_DIM).transpose(1,2)
        v = self.v(x).view(B,T,N_KV_HEAD,HEAD_DIM).transpose(1,2)
        q, k = rope(q), rope(k)
        k = k.repeat_interleave(N_HEAD//N_KV_HEAD, dim=1)
        v = v.repeat_interleave(N_HEAD//N_KV_HEAD, dim=1)
        y = F.scaled_dot_product_attention(q,k,v,is_causal=True)
        y = y.transpose(1,2).contiguous().view(B,T,C)
        return self.drop(self.proj(y))

class SwiGLU(nn.Module):
    def __init__(self):
        super().__init__(); hidden = 4*N_EMBD
        self.gate = nn.Linear(N_EMBD, hidden, bias=False)
        self.up = nn.Linear(N_EMBD, hidden, bias=False)
        self.down = nn.Linear(hidden, N_EMBD, bias=False)
    def forward(self,x): return self.down(F.silu(self.gate(x))*self.up(x))

class MoE(nn.Module):
    def __init__(self):
        super().__init__()
        self.router = nn.Linear(N_EMBD, N_EXPERTS, bias=False)
        self.experts = nn.ModuleList([SwiGLU() for _ in range(N_EXPERTS)])
        # Эксперты стартуют как копии одной FFN с малой диверсификацией.
        base = copy.deepcopy(self.experts[0].state_dict())
        for i in range(1,N_EXPERTS):
            self.experts[i].load_state_dict(base)
            with torch.no_grad():
                for p in self.experts[i].parameters(): p.add_(torch.randn_like(p)*0.005)
    def forward(self,x):
        logits = self.router(x)
        vals, inds = logits.topk(TOP_K, dim=-1)
        weights = F.softmax(vals, dim=-1)
        out = torch.zeros_like(x)
        for slot in range(TOP_K):
            idx = inds[...,slot]
            w = weights[...,slot:slot+1]
            for e, expert in enumerate(self.experts):
                mask = (idx == e).unsqueeze(-1)
                if mask.any(): out = out + expert(x) * (w * mask)
        return out

class Block(nn.Module):
    def __init__(self):
        super().__init__(); self.n1=RMSNorm(N_EMBD); self.attn=GQAAttention(); self.n2=RMSNorm(N_EMBD); self.moe=MoE()
    def forward(self,x):
        x = x + self.attn(self.n1(x)); x = x + self.moe(self.n2(x)); return x

class KOTMini(nn.Module):
    def __init__(self):
        super().__init__(); self.token=nn.Embedding(vocab_size,N_EMBD); self.blocks=nn.ModuleList([Block() for _ in range(N_LAYER)]); self.norm=RMSNorm(N_EMBD); self.head=nn.Linear(N_EMBD,vocab_size,bias=False); self.head.weight=self.token.weight
    def forward(self,idx,targets=None):
        x=self.token(idx)
        for block in self.blocks: x=block(x)
        logits=self.head(self.norm(x)); loss=None
        if targets is not None: loss=F.cross_entropy(logits.reshape(-1,vocab_size),targets.reshape(-1))
        return logits,loss
    @torch.no_grad()
    def generate(self,idx,max_new_tokens=350,temperature=0.85,top_k=40):
        self.eval()
        for _ in range(max_new_tokens):
            x=idx[:,-BLOCK_SIZE:]
            logits,_=self(x); logits=logits[:,-1,:]/max(temperature,1e-4)
            if top_k: 
                v,_=torch.topk(logits,min(top_k,logits.size(-1))); logits[logits < v[:,-1,None]]=-float('inf')
            probs=F.softmax(logits,dim=-1); nxt=torch.multinomial(probs,1); idx=torch.cat((idx,nxt),dim=1)
        return idx

model=KOTMini().to(device)
print(f'Параметров: {sum(p.numel() for p in model.parameters()):,}')


## 3. Обучение

FP16 autocast + GradScaler на CUDA. Данные режутся на блоки без padding. Сохраняем лучший checkpoint по validation loss.

In [ ]:
def get_batch(split_name):
    data = train_ids if split_name=='train' else val_ids
    ix = torch.randint(len(data)-BLOCK_SIZE-1,(BATCH_SIZE,))
    x=torch.stack([data[i:i+BLOCK_SIZE] for i in ix]); y=torch.stack([data[i+1:i+BLOCK_SIZE+1] for i in ix])
    return x.to(device),y.to(device)

@torch.no_grad()
def estimate_loss():
    model.eval(); out={}
    for name in ['train','val']:
        losses=[]
        for _ in range(EVAL_BATCHES):
            xb,yb=get_batch(name)
            with torch.autocast('cuda',dtype=torch.float16): _,loss=model(xb,yb)
            losses.append(float(loss))
        out[name]=sum(losses)/len(losses)
    model.train(); return out

optimizer=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=0.1)
scaler=torch.amp.GradScaler('cuda')
history={'step':[],'train':[],'val':[]}
best_val=float('inf'); best_state=None
start=time.time(); model.train()
for step in range(1,MAX_STEPS+1):
    xb,yb=get_batch('train'); optimizer.zero_grad(set_to_none=True)
    with torch.autocast('cuda',dtype=torch.float16): _,loss=model(xb,yb)
    scaler.scale(loss).backward(); scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
    scaler.step(optimizer); scaler.update()
    if step==1 or step%EVAL_EVERY==0 or step==MAX_STEPS:
        metrics=estimate_loss(); history['step'].append(step); history['train'].append(metrics['train']); history['val'].append(metrics['val'])
        print(f"step {step:4d}/{MAX_STEPS} | train {metrics['train']:.4f} | val {metrics['val']:.4f} | {time.time()-start:.1f}s")
        if metrics['val'] < best_val:
            best_val=metrics['val']; best_state=copy.deepcopy(model.state_dict())
model.load_state_dict(best_state); model.eval()
print('Обучение завершено. Лучший val loss:', round(best_val,4))
plt.figure(figsize=(8,4)); plt.plot(history['step'],history['train'],label='train'); plt.plot(history['step'],history['val'],label='validation'); plt.xlabel('Step'); plt.ylabel('Cross-entropy loss'); plt.legend(); plt.grid(True); plt.show()


## 4. Ползунок изменения весов экспертов

Здесь `0.0` означает: все четыре эксперта заменены их средними весами (полная унификация). `1.0` — исходные веса после обучения. Значения выше `1.0` усиливают различия между экспертами (экстраполяция). Это контролируемое изменение **реальных параметров** FFN, а не просто настройка температуры генерации.

После отпускания ползунка код восстанавливает исходный checkpoint, применяет выбранную степень расхождения и заново считает validation loss на фиксированных батчах, затем генерирует рассказ.

In [ ]:
# Сохраняем неизменённый checkpoint и фиксированные validation batches для честного сравнения.
trained_state=copy.deepcopy(model.state_dict())
torch.manual_seed(2026)
fixed_val_batches=[get_batch('val') for _ in range(10)]
prompt_box=widgets.Textarea(value='Однажды маленький котёнок вышел во двор и увидел',description='Промпт:',layout=widgets.Layout(width='95%',height='70px'))
strength=widgets.FloatSlider(value=1.0,min=0.0,max=2.0,step=0.05,description='Изменение:',readout_format='.2f',continuous_update=False,style={'description_width':'initial'},layout=widgets.Layout(width='95%'))
out=widgets.Output()

expert_keys=[]
for li in range(N_LAYER):
    for ei in range(N_EXPERTS):
        prefix=f'blocks.{li}.moe.experts.{ei}.'
        expert_keys.append((li,ei,prefix))

def apply_expert_strength(alpha):
    model.load_state_dict(trained_state)
    with torch.no_grad():
        for li in range(N_LAYER):
            for suffix in ['gate.weight','up.weight','down.weight']:
                params=[dict(model.named_parameters())[f'blocks.{li}.moe.experts.{ei}.{suffix}'] for ei in range(N_EXPERTS)]
                mean=torch.stack([p.data for p in params]).mean(0)
                for p in params: p.data.copy_(mean + alpha*(p.data-mean))

@torch.no_grad()
def fixed_validation_loss():
    model.eval(); vals=[]
    for xb,yb in fixed_val_batches:
        with torch.autocast('cuda',dtype=torch.float16): _,l=model(xb,yb)
        vals.append(float(l))
    return sum(vals)/len(vals)

def on_change(change=None):
    alpha=float(strength.value)
    with out:
        clear_output(wait=True)
        apply_expert_strength(alpha)
        loss=fixed_validation_loss()
        p=prompt_box.value.strip() or 'Однажды в маленьком лесу'
        # Неизвестные символы заменяем пробелом.
        ids=[stoi.get(c,stoi.get(' ',0)) for c in p]
        context=torch.tensor([ids],dtype=torch.long,device=device)
        generated=model.generate(context,max_new_tokens=260,temperature=0.85,top_k=40)[0].tolist()
        story=''.join(itos.get(i,'') for i in generated)
        print(f'Сила расхождения экспертов: {alpha:.2f}')
        print(f'Validation loss: {loss:.4f} | perplexity: {math.exp(min(loss,20)):.2f}')
        print('\n--- Генерация ---\n')
        print(story)
        print('\n--- Что означает ползунок ---')
        print('0.00 = эксперты сведены к среднему; 1.00 = исходные обученные веса; 2.00 = различия усилены в 2 раза.')

strength.observe(on_change,names='value')
display(prompt_box,strength,out)
on_change()


## 5. Как читать результат

- Сравнивай loss при `0.00`, `1.00` и `2.00`.
- Если loss минимален около `1.00`, исходная специализация экспертов полезна.
- Если `0.00` лучше, эксперты могли не успеть специализироваться или routing оказался нестабильным.
- Если `2.00` лучше, усиление различий случайно помогло на этой validation выборке; это ещё не доказательство улучшения генерации в целом.

Loss считается на одном и том же наборе validation-батчей, чтобы значения можно было сравнивать. Генерация стохастическая, поэтому для строгого сравнения качества лучше дополнительно использовать одинаковый seed или несколько запусков.

**Эксперимент ограничен:** символьная токенизация, небольшой контекст 256 и короткое обучение. Он показывает механику копирования/расхождения весов, но не предсказывает качество большой KOT-V2.

In [ ]:
# Скачать checkpoint мини-модели (необязательно)
torch.save({'model':trained_state,'stoi':stoi,'itos':itos,'config':{'n_layer':N_LAYER,'n_embd':N_EMBD,'n_head':N_HEAD,'n_kv_head':N_KV_HEAD,'n_experts':N_EXPERTS,'top_k':TOP_K,'block_size':BLOCK_SIZE}},'KOT_V2_Mini_RU.pt')
from google.colab import files
files.download('KOT_V2_Mini_RU.pt')
